# ARC DSL Program Synthesis Baseline

**Competition:** ARC-Prize-2026 / ARC-AGI-2  
**Approach:** Program synthesis via BFS over a hand-crafted DSL  
**Metric:** pass@2 — a task is solved if *either* of our two attempts matches the hidden output

---

## What this notebook does

We treat every ARC task as a *program search* problem: given a small set of
demonstration pairs (input→output grids), find a short composition of grid
transformations that explains all demonstrations, then apply that program to the
test input.

The solver has two rungs:

### Rung 1 — trivial baseline (`solve_trivial`)
Used as a fallback when no program is found.  
- Attempt 1: identity (output = input)  
- Attempt 2: the constant demo output (if all demos agree) or a zero-grid

### Rung 2B — DSL + object-level search (`solve_dsl_obj`)

**DSL (30 whole-grid ops)**  
A library of pure `Grid → Grid` functions covering the D8 symmetry group
(8 rotations/flips), scaling & tiling (upscale 2×/3×, tile, compress, mirror),
cropping & selection (bounding box, border removal, quadrants), color remapping
(inferred per-task pixel permutation, swap dominants, keep dominant), and
physics-ish ops (column/row gravity, outline extraction).

**Object ops (~117 generated closures)**  
Three segmentation strategies (4-connected same-color, 8-connected same-color,
4-connected any-non-background) × 9 selectors (largest, smallest, topmost,
bottommost, leftmost, rightmost, unique-color, unique-shape, most-common-shape)
× 4 actions (keep-only, remove, crop-to-bounding-box, recolor-to-dominant) = 108
selector-action closures, plus 3 segmentations × whole-decomp transforms
(recolor-by-size-rank, gravity in 4 directions, count-as-grid) = 9 more.

**BFS with pruning**  
Iterative-deepening BFS over op compositions up to depth 3.  
Three pruning mechanisms keep it tractable:
1. *Shape gate* — SHAPE_RULES predicts output dimensions; programs whose predicted
   shape disagrees with demo output shape are pruned immediately.
2. *State dedup* — at each depth level, if two different op sequences produce the
   same intermediate grid on the first demo input, only one branch is explored.
3. *First-demo short-circuit* — full verification against all demo pairs only
   happens when the first demo output matches.

For depth-3 with object ops, search is restricted to mixed whole/object patterns
(WOW, OWW, WWO) to avoid the combinatorial blowup of full OOO/OWO depth-3.
A segmentation-consistency gate further restricts which segmentations are active
per task: a segmentation is kept only if ≥50 % of demo pairs show a meaningful
relationship (non-trivial object count on both sides, color overlap, same count,
or close count).

**pass@2 via distinct-hypothesis ranking**  
All consistent programs are grouped by the grid they predict on each test input.
Groups are ranked by (shortest program length in group ascending, group size
descending). Attempt 1 = top-ranked group's grid; Attempt 2 = second-ranked
group's grid (or rung-1 fallback if there is only one hypothesis).

**Honest framing**  
This is an interpretable *baseline* — a first rung of a longer ladder toward
general ARC solving. The DSL covers a meaningful slice of ARC's transformation
vocabulary; the BFS is exact within depth 3. Harder tasks requiring spatial
reasoning beyond simple compositions, conditional logic, or multi-step object
interactions are outside this solver's scope. Future rungs would extend the
object action vocabulary, add inductive logic programming over object-relation
predicates, or use neural guidance to focus search.


In [ ]:
# ============================================================
# Stdlib imports (all modules combined, deduplicated)
# ============================================================
from __future__ import annotations

import json
import logging
import time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("arc")
print("Imports OK")

In [ ]:
# ============================================================
# arc_io — data loading, types, submission builder
# ============================================================

Grid = list[list[int]]


@dataclass(frozen=True)
class Pair:
    """One demonstration pair: input grid -> expected output grid."""
    input: Grid
    output: Grid


@dataclass(frozen=True)
class Task:
    """One ARC task: demo pairs plus test inputs (solutions when available)."""
    task_id: str
    train: list[Pair]
    test_inputs: list[Grid]
    test_outputs: list[Grid] | None = field(default=None)  # None for hidden test set


def build_submission(
    attempts: dict[str, list[tuple[Grid, Grid]]], out_path: Path
) -> None:
    """Write submission JSON: task_id -> [{attempt_1, attempt_2}, ...] per test input."""
    payload = {
        task_id: [{"attempt_1": a1, "attempt_2": a2} for a1, a2 in per_input]
        for task_id, per_input in attempts.items()
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(payload))
    log.info("wrote %s (%d tasks)", out_path, len(payload))


print("arc_io OK")

In [ ]:
# ============================================================
# dsl — ~30 pure Grid->Grid operations
# ============================================================

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _copy(grid: Grid) -> Grid:
    """Return a deep copy of the grid."""
    return [row[:] for row in grid]


def _rows(grid: Grid) -> int:
    return len(grid)


def _cols(grid: Grid) -> int:
    return len(grid[0]) if grid else 0


def _dominant_nonzero(grid: Grid) -> int:
    """Return the most frequent non-zero color; 0 if grid is all zeros."""
    counts: Counter[int] = Counter()
    for row in grid:
        for v in row:
            if v != 0:
                counts[v] += 1
    if not counts:
        return 0
    return counts.most_common(1)[0][0]


# ---------------------------------------------------------------------------
# Geometry (8 ops) — D8 symmetry group
# ---------------------------------------------------------------------------

def identity(grid: Grid) -> Grid:
    """Pass the grid through unchanged — identity element of composition."""
    return _copy(grid)


def rotate90(grid: Grid) -> Grid:
    """Rotate 90 degrees clockwise."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    result = [[grid[R - 1 - c][r] for c in range(R)] for r in range(C)]
    return result


def rotate180(grid: Grid) -> Grid:
    """Rotate 180 degrees."""
    return [row[::-1] for row in grid[::-1]]


def rotate270(grid: Grid) -> Grid:
    """Rotate 270 degrees clockwise (= 90 degrees counter-clockwise)."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    result = [[grid[c][C - 1 - r] for c in range(R)] for r in range(C)]
    return result


def flip_horizontal(grid: Grid) -> Grid:
    """Mirror left<->right."""
    return [row[::-1] for row in grid]


def flip_vertical(grid: Grid) -> Grid:
    """Mirror top<->bottom."""
    return grid[::-1]


def transpose(grid: Grid) -> Grid:
    """Transpose along main diagonal."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    return [[grid[r][c] for r in range(R)] for c in range(C)]


def anti_transpose(grid: Grid) -> Grid:
    """Transpose along anti-diagonal."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    return [[grid[R - 1 - j][C - 1 - i] for j in range(R)] for i in range(C)]


# ---------------------------------------------------------------------------
# Scaling & tiling (8 ops)
# ---------------------------------------------------------------------------

def upscale_2x(grid: Grid) -> Grid:
    """Scale up 2x by duplicating every cell into a 2x2 block."""
    result = []
    for row in grid:
        new_row = [v for v in row for _ in range(2)]
        result.append(new_row)
        result.append(new_row[:])
    return result


def upscale_3x(grid: Grid) -> Grid:
    """Scale up 3x by duplicating every cell into a 3x3 block."""
    result = []
    for row in grid:
        new_row = [v for v in row for _ in range(3)]
        for _ in range(3):
            result.append(new_row[:])
    return result


def tile_2x2(grid: Grid) -> Grid:
    """Tile the grid in a 2x2 arrangement."""
    result = []
    for row in grid:
        result.append(row + row)
    for row in grid:
        result.append(row + row)
    return result


def tile_3x3(grid: Grid) -> Grid:
    """Tile the grid in a 3x3 arrangement."""
    result = []
    for _ in range(3):
        for row in grid:
            result.append(row * 3)
    return result


def mirror_tile_right(grid: Grid) -> Grid:
    """Tile horizontally as [grid | h-flip(grid)]."""
    flipped = flip_horizontal(grid)
    return [r1 + r2 for r1, r2 in zip(grid, flipped)]


def mirror_tile_down(grid: Grid) -> Grid:
    """Tile vertically as [grid / v-flip(grid)]."""
    return _copy(grid) + flip_vertical(grid)


def compress_2x(grid: Grid) -> Grid:
    """Inverse of upscale_2x: collapse 2x2 uniform blocks into single cells."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0 or R % 2 != 0 or C % 2 != 0:
        return _copy(grid)
    result = []
    for br in range(0, R, 2):
        row_out = []
        for bc in range(0, C, 2):
            vals = {grid[br + dr][bc + dc] for dr in range(2) for dc in range(2)}
            if len(vals) != 1:
                return _copy(grid)
            row_out.append(vals.pop())
        result.append(row_out)
    return result


def deduplicate_rows_cols(grid: Grid) -> Grid:
    """Remove consecutive duplicate rows, then consecutive duplicate columns."""
    deduped_rows: list[list[int]] = []
    for row in grid:
        if not deduped_rows or row != deduped_rows[-1]:
            deduped_rows.append(row)
    if not deduped_rows:
        return _copy(grid)
    C = len(deduped_rows[0])
    keep_cols: list[int] = []
    prev_col: list[int] | None = None
    for c in range(C):
        col = [deduped_rows[r][c] for r in range(len(deduped_rows))]
        if col != prev_col:
            keep_cols.append(c)
            prev_col = col
    return [[row[c] for c in keep_cols] for row in deduped_rows]


# ---------------------------------------------------------------------------
# Cropping & selection (7 ops)
# ---------------------------------------------------------------------------

def crop_to_content(grid: Grid) -> Grid:
    """Bounding box of non-zero cells; return input if grid is all zeros."""
    R, C = _rows(grid), _cols(grid)
    rows_with: list[int] = [r for r in range(R) if any(grid[r][c] != 0 for c in range(C))]
    cols_with: list[int] = [c for c in range(C) if any(grid[r][c] != 0 for r in range(R))]
    if not rows_with or not cols_with:
        return _copy(grid)
    r0, r1 = rows_with[0], rows_with[-1]
    c0, c1 = cols_with[0], cols_with[-1]
    return [grid[r][c0:c1 + 1] for r in range(r0, r1 + 1)]


def remove_border(grid: Grid) -> Grid:
    """Strip the 1-cell outer frame; return input if grid is too small."""
    R, C = _rows(grid), _cols(grid)
    if R <= 2 or C <= 2:
        return _copy(grid)
    return [grid[r][1:C - 1] for r in range(1, R - 1)]


def top_left_quadrant(grid: Grid) -> Grid:
    """Return the top-left quadrant of the grid."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    return [grid[r][:C // 2] for r in range(R // 2)]


def top_right_quadrant(grid: Grid) -> Grid:
    """Return the top-right quadrant of the grid."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    return [grid[r][C // 2:] for r in range(R // 2)]


def bottom_left_quadrant(grid: Grid) -> Grid:
    """Return the bottom-left quadrant of the grid."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    return [grid[r][:C // 2] for r in range(R // 2, R)]


def bottom_right_quadrant(grid: Grid) -> Grid:
    """Return the bottom-right quadrant of the grid."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    return [grid[r][C // 2:] for r in range(R // 2, R)]


def largest_color_rectangle(grid: Grid) -> Grid:
    """Bounding box of the most frequent non-zero color."""
    dominant = _dominant_nonzero(grid)
    if dominant == 0:
        return _copy(grid)
    R, C = _rows(grid), _cols(grid)
    rows_with = [r for r in range(R) if any(grid[r][c] == dominant for c in range(C))]
    cols_with = [c for c in range(C) if any(grid[r][c] == dominant for r in range(R))]
    if not rows_with or not cols_with:
        return _copy(grid)
    r0, r1 = rows_with[0], rows_with[-1]
    c0, c1 = cols_with[0], cols_with[-1]
    return [grid[r][c0:c1 + 1] for r in range(r0, r1 + 1)]


# ---------------------------------------------------------------------------
# Color ops (4 ops)
# ---------------------------------------------------------------------------

def swap_two_dominant_colors(grid: Grid) -> Grid:
    """Swap the two most frequent non-zero colors."""
    counts: Counter[int] = Counter(v for row in grid for v in row if v != 0)
    if len(counts) < 2:
        return _copy(grid)
    top2 = [c for c, _ in counts.most_common(2)]
    a, b = top2[0], top2[1]
    mapping = {a: b, b: a}
    return [[mapping.get(v, v) for v in row] for row in grid]


def replace_background_with_most_common(grid: Grid) -> Grid:
    """Replace all 0s with the most frequent non-zero color."""
    fill = _dominant_nonzero(grid)
    if fill == 0:
        return _copy(grid)
    return [[fill if v == 0 else v for v in row] for row in grid]


def keep_dominant_color_only(grid: Grid) -> Grid:
    """Zero out all non-zero cells except the most frequent non-zero color."""
    dominant = _dominant_nonzero(grid)
    if dominant == 0:
        return _copy(grid)
    return [[v if v == dominant else 0 for v in row] for row in grid]


def infer_color_map(grid: Grid) -> Grid:
    """Placeholder — replaced per-task by build_color_map_op(); returns identity."""
    return _copy(grid)


def build_color_map_op(pairs: list[tuple[Grid, Grid]]) -> Callable[[Grid], Grid]:
    """Factory: infer a pixel-wise color permutation from demo pairs."""
    mapping: dict[int, int] = {}
    for inp, out in pairs:
        if _rows(inp) != _rows(out) or _cols(inp) != _cols(out):
            return _copy  # type: ignore[return-value]
        for row_i, row_o in zip(inp, out):
            for vi, vo in zip(row_i, row_o):
                if vi in mapping:
                    if mapping[vi] != vo:
                        return _copy  # type: ignore[return-value]
                else:
                    mapping[vi] = vo

    if not mapping:
        return _copy  # type: ignore[return-value]

    def _apply(grid: Grid) -> Grid:
        return [[mapping.get(v, v) for v in row] for row in grid]

    return _apply


# ---------------------------------------------------------------------------
# Physics-ish (3 ops)
# ---------------------------------------------------------------------------

def gravity_down(grid: Grid) -> Grid:
    """Non-zero cells fall to the bottom of each column."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    result = [[0] * C for _ in range(R)]
    for c in range(C):
        col = [grid[r][c] for r in range(R)]
        nonzero = [v for v in col if v != 0]
        zeros = [0] * (R - len(nonzero))
        new_col = zeros + nonzero
        for r in range(R):
            result[r][c] = new_col[r]
    return result


def gravity_left(grid: Grid) -> Grid:
    """Non-zero cells fall to the left of each row."""
    result = []
    for row in grid:
        nonzero = [v for v in row if v != 0]
        zeros = [0] * (len(row) - len(nonzero))
        result.append(nonzero + zeros)
    return result


def outline_objects(grid: Grid) -> Grid:
    """Keep only boundary cells of non-zero regions; interior cells become 0."""
    R, C = _rows(grid), _cols(grid)
    if R == 0 or C == 0:
        return _copy(grid)
    result = [[0] * C for _ in range(R)]
    for r in range(R):
        for c in range(C):
            if grid[r][c] == 0:
                continue
            is_boundary = False
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                nr, nc = r + dr, c + dc
                if nr < 0 or nr >= R or nc < 0 or nc >= C or grid[nr][nc] == 0:
                    is_boundary = True
                    break
            if is_boundary:
                result[r][c] = grid[r][c]
    return result


# ---------------------------------------------------------------------------
# OPS registry
# ---------------------------------------------------------------------------

OPS: dict[str, Callable[[Grid], Grid]] = {
    "identity": identity,
    "rotate90": rotate90,
    "rotate180": rotate180,
    "rotate270": rotate270,
    "flip_horizontal": flip_horizontal,
    "flip_vertical": flip_vertical,
    "transpose": transpose,
    "anti_transpose": anti_transpose,
    "upscale_2x": upscale_2x,
    "upscale_3x": upscale_3x,
    "tile_2x2": tile_2x2,
    "tile_3x3": tile_3x3,
    "mirror_tile_right": mirror_tile_right,
    "mirror_tile_down": mirror_tile_down,
    "compress_2x": compress_2x,
    "deduplicate_rows_cols": deduplicate_rows_cols,
    "crop_to_content": crop_to_content,
    "remove_border": remove_border,
    "top_left_quadrant": top_left_quadrant,
    "top_right_quadrant": top_right_quadrant,
    "bottom_left_quadrant": bottom_left_quadrant,
    "bottom_right_quadrant": bottom_right_quadrant,
    "largest_color_rectangle": largest_color_rectangle,
    "infer_color_map": infer_color_map,
    "swap_two_dominant_colors": swap_two_dominant_colors,
    "replace_background_with_most_common": replace_background_with_most_common,
    "keep_dominant_color_only": keep_dominant_color_only,
    "gravity_down": gravity_down,
    "gravity_left": gravity_left,
    "outline_objects": outline_objects,
}


# ---------------------------------------------------------------------------
# SHAPE_RULES
# ---------------------------------------------------------------------------

def _shape_identity(r: int, c: int) -> tuple[int, int]:
    return (r, c)


def _shape_rotate(r: int, c: int) -> tuple[int, int]:
    return (c, r)


def _shape_upscale2(r: int, c: int) -> tuple[int, int]:
    return (r * 2, c * 2)


def _shape_upscale3(r: int, c: int) -> tuple[int, int]:
    return (r * 3, c * 3)


def _shape_tile2x2(r: int, c: int) -> tuple[int, int]:
    return (r * 2, c * 2)


def _shape_tile3x3(r: int, c: int) -> tuple[int, int]:
    return (r * 3, c * 3)


def _shape_mirror_right(r: int, c: int) -> tuple[int, int]:
    return (r, c * 2)


def _shape_mirror_down(r: int, c: int) -> tuple[int, int]:
    return (r * 2, c)


def _shape_compress2(r: int, c: int) -> tuple[int, int] | None:
    if r % 2 == 0 and c % 2 == 0:
        return (r // 2, c // 2)
    return None


def _shape_remove_border(r: int, c: int) -> tuple[int, int] | None:
    if r > 2 and c > 2:
        return (r - 2, c - 2)
    return None


def _shape_quadrant(r: int, c: int) -> tuple[int, int]:
    return (r // 2, c // 2)


SHAPE_RULES: dict[str, Callable[[int, int], tuple[int, int] | None]] = {
    "identity": _shape_identity,
    "rotate90": _shape_rotate,
    "rotate180": _shape_identity,
    "rotate270": _shape_rotate,
    "flip_horizontal": _shape_identity,
    "flip_vertical": _shape_identity,
    "transpose": _shape_rotate,
    "anti_transpose": _shape_rotate,
    "upscale_2x": _shape_upscale2,
    "upscale_3x": _shape_upscale3,
    "tile_2x2": _shape_tile2x2,
    "tile_3x3": _shape_tile3x3,
    "mirror_tile_right": _shape_mirror_right,
    "mirror_tile_down": _shape_mirror_down,
    "compress_2x": _shape_compress2,
    "deduplicate_rows_cols": None,
    "crop_to_content": None,
    "remove_border": _shape_remove_border,
    "top_left_quadrant": _shape_quadrant,
    "top_right_quadrant": _shape_quadrant,
    "bottom_left_quadrant": _shape_quadrant,
    "bottom_right_quadrant": _shape_quadrant,
    "largest_color_rectangle": None,
    "infer_color_map": _shape_identity,
    "swap_two_dominant_colors": _shape_identity,
    "replace_background_with_most_common": _shape_identity,
    "keep_dominant_color_only": _shape_identity,
    "gravity_down": _shape_identity,
    "gravity_left": _shape_identity,
    "outline_objects": _shape_identity,
}


print(f"DSL OK — {len(OPS)} ops registered")

In [ ]:
# ============================================================
# objects — object segmentation, selectors, actions, op generation
# ============================================================

# ---------------------------------------------------------------------------
# Obj dataclass
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Obj:
    """An object extracted from an ARC grid."""
    cells: frozenset[tuple[int, int]]
    color: int

    def size(self) -> int:
        return len(self.cells)

    def bbox(self) -> tuple[int, int, int, int]:
        rows = [r for r, c in self.cells]
        cols = [c for r, c in self.cells]
        return (min(rows), min(cols), max(rows), max(cols))

    def height(self) -> int:
        r0, c0, r1, c1 = self.bbox()
        return r1 - r0 + 1

    def width(self) -> int:
        r0, c0, r1, c1 = self.bbox()
        return c1 - c0 + 1

    def normalized_shape(self) -> frozenset[tuple[int, int]]:
        r0, c0, _, _ = self.bbox()
        return frozenset((r - r0, c - c0) for r, c in self.cells)


# ---------------------------------------------------------------------------
# Background detection
# ---------------------------------------------------------------------------

def background_color(grid: Grid) -> int:
    """Return the most frequent color (background); ties broken by lowest value."""
    counts: Counter[int] = Counter(v for row in grid for v in row)
    if not counts:
        return 0
    max_count = max(counts.values())
    candidates = sorted(c for c, n in counts.items() if n == max_count)
    return candidates[0]


# ---------------------------------------------------------------------------
# Segmentation helpers
# ---------------------------------------------------------------------------

def _flood_fill_4(
    grid: Grid,
    start_r: int,
    start_c: int,
    visited: set[tuple[int, int]],
    match_fn: Callable[[int, int], bool],
) -> list[tuple[int, int]]:
    R = len(grid)
    C = len(grid[0]) if grid else 0
    stack = [(start_r, start_c)]
    component: list[tuple[int, int]] = []
    while stack:
        r, c = stack.pop()
        if (r, c) in visited:
            continue
        if r < 0 or r >= R or c < 0 or c >= C:
            continue
        if not match_fn(r, c):
            continue
        visited.add((r, c))
        component.append((r, c))
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited:
                stack.append((nr, nc))
    return component


def _flood_fill_8(
    grid: Grid,
    start_r: int,
    start_c: int,
    visited: set[tuple[int, int]],
    match_fn: Callable[[int, int], bool],
) -> list[tuple[int, int]]:
    R = len(grid)
    C = len(grid[0]) if grid else 0
    stack = [(start_r, start_c)]
    component: list[tuple[int, int]] = []
    while stack:
        r, c = stack.pop()
        if (r, c) in visited:
            continue
        if r < 0 or r >= R or c < 0 or c >= C:
            continue
        if not match_fn(r, c):
            continue
        visited.add((r, c))
        component.append((r, c))
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if (nr, nc) not in visited:
                    stack.append((nr, nc))
    return component


def _obj_from_cells(grid: Grid, cells: list[tuple[int, int]]) -> Obj:
    counts: Counter[int] = Counter(grid[r][c] for r, c in cells)
    color = counts.most_common(1)[0][0] if counts else 0
    return Obj(cells=frozenset(cells), color=color)


# ---------------------------------------------------------------------------
# Segmentation functions
# ---------------------------------------------------------------------------

def objects_4conn_same_color(grid: Grid) -> list[Obj]:
    """4-connected same-color segmentation (background excluded)."""
    if not grid or not grid[0]:
        return []
    bg = background_color(grid)
    R, C = len(grid), len(grid[0])
    visited: set[tuple[int, int]] = set()
    objects: list[Obj] = []
    for r in range(R):
        for c in range(C):
            if (r, c) in visited or grid[r][c] == bg:
                continue
            color = grid[r][c]
            cells = _flood_fill_4(
                grid, r, c, visited,
                lambda rr, cc, col=color: grid[rr][cc] == col,
            )
            if cells:
                objects.append(Obj(cells=frozenset(cells), color=color))
    return objects


def objects_8conn_same_color(grid: Grid) -> list[Obj]:
    """8-connected same-color segmentation (background excluded)."""
    if not grid or not grid[0]:
        return []
    bg = background_color(grid)
    R, C = len(grid), len(grid[0])
    visited: set[tuple[int, int]] = set()
    objects: list[Obj] = []
    for r in range(R):
        for c in range(C):
            if (r, c) in visited or grid[r][c] == bg:
                continue
            color = grid[r][c]
            cells = _flood_fill_8(
                grid, r, c, visited,
                lambda rr, cc, col=color: grid[rr][cc] == col,
            )
            if cells:
                objects.append(Obj(cells=frozenset(cells), color=color))
    return objects


def objects_multicolor_4conn(grid: Grid) -> list[Obj]:
    """4-connected any-non-background segmentation (composite objects)."""
    if not grid or not grid[0]:
        return []
    bg = background_color(grid)
    R, C = len(grid), len(grid[0])
    visited: set[tuple[int, int]] = set()
    objects: list[Obj] = []
    for r in range(R):
        for c in range(C):
            if (r, c) in visited or grid[r][c] == bg:
                continue
            cells = _flood_fill_4(
                grid, r, c, visited,
                lambda rr, cc: grid[rr][cc] != bg,
            )
            if cells:
                objects.append(_obj_from_cells(grid, cells))
    return objects


SEGMENTATIONS: dict[str, Callable[[Grid], list[Obj]]] = {
    "4conn": objects_4conn_same_color,
    "8conn": objects_8conn_same_color,
    "multi": objects_multicolor_4conn,
}


# ---------------------------------------------------------------------------
# Selectors
# ---------------------------------------------------------------------------

def _sel_largest(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    return max(objs, key=lambda o: o.size())


def _sel_smallest(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    return min(objs, key=lambda o: o.size())


def _sel_most_common_shape(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    shape_counts: Counter[frozenset[tuple[int, int]]] = Counter(o.normalized_shape() for o in objs)
    most_common_shape = shape_counts.most_common(1)[0][0]
    for o in objs:
        if o.normalized_shape() == most_common_shape:
            return o
    return None


def _sel_unique_shape(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    shape_counts: Counter[frozenset[tuple[int, int]]] = Counter(o.normalized_shape() for o in objs)
    unique_shapes = {s for s, n in shape_counts.items() if n == 1}
    for o in objs:
        if o.normalized_shape() in unique_shapes:
            return o
    return None


def _sel_unique_color(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    color_counts: Counter[int] = Counter(o.color for o in objs)
    unique_colors = {c for c, n in color_counts.items() if n == 1}
    for o in objs:
        if o.color in unique_colors:
            return o
    return None


def _sel_topmost(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    return min(objs, key=lambda o: o.bbox()[0])


def _sel_bottommost(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    return max(objs, key=lambda o: o.bbox()[2])


def _sel_leftmost(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    return min(objs, key=lambda o: o.bbox()[1])


def _sel_rightmost(objs: list[Obj]) -> Obj | None:
    if not objs: return None
    return max(objs, key=lambda o: o.bbox()[3])


SELECTORS: dict[str, Callable[[list[Obj]], Obj | None]] = {
    "largest": _sel_largest,
    "smallest": _sel_smallest,
    "most_common_shape": _sel_most_common_shape,
    "unique_shape": _sel_unique_shape,
    "unique_color": _sel_unique_color,
    "topmost": _sel_topmost,
    "bottommost": _sel_bottommost,
    "leftmost": _sel_leftmost,
    "rightmost": _sel_rightmost,
}


# ---------------------------------------------------------------------------
# Actions on a selected object
# ---------------------------------------------------------------------------

def _action_keep_only(grid: Grid, obj: Obj, seg_fn: Callable[[Grid], list[Obj]]) -> Grid:
    if not grid or not grid[0]: return [row[:] for row in grid]
    bg = background_color(grid)
    R, C = len(grid), len(grid[0])
    result = [[bg] * C for _ in range(R)]
    for r, c in obj.cells:
        result[r][c] = grid[r][c]
    return result


def _action_remove(grid: Grid, obj: Obj, seg_fn: Callable[[Grid], list[Obj]]) -> Grid:
    if not grid or not grid[0]: return [row[:] for row in grid]
    bg = background_color(grid)
    result = [row[:] for row in grid]
    for r, c in obj.cells:
        result[r][c] = bg
    return result


def _action_crop_to(grid: Grid, obj: Obj, seg_fn: Callable[[Grid], list[Obj]]) -> Grid:
    r0, c0, r1, c1 = obj.bbox()
    return [grid[r][c0:c1 + 1] for r in range(r0, r1 + 1)]


def _action_recolor_to_dominant(grid: Grid, obj: Obj, seg_fn: Callable[[Grid], list[Obj]]) -> Grid:
    if not grid or not grid[0]: return [row[:] for row in grid]
    bg = background_color(grid)
    R, C = len(grid), len(grid[0])
    result = [[bg] * C for _ in range(R)]
    for r in range(R):
        for c in range(C):
            if grid[r][c] != bg:
                result[r][c] = obj.color
    return result


# ---------------------------------------------------------------------------
# Whole-decomposition transforms
# ---------------------------------------------------------------------------

def _recolor_by_size_rank(grid: Grid, seg_fn: Callable[[Grid], list[Obj]]) -> Grid:
    if not grid or not grid[0]: return [row[:] for row in grid]
    bg = background_color(grid)
    R, C = len(grid), len(grid[0])
    objs = seg_fn(grid)
    if not objs: return [row[:] for row in grid]
    sorted_objs = sorted(objs, key=lambda o: o.size(), reverse=True)
    result = [[bg] * C for _ in range(R)]
    for rank, obj in enumerate(sorted_objs, start=1):
        new_color = rank % 10
        for r, c in obj.cells:
            result[r][c] = new_color
    return result


def _move_all_to_gravity(grid: Grid, seg_fn: Callable[[Grid], list[Obj]], direction: str = "down") -> Grid:
    if not grid or not grid[0]: return [row[:] for row in grid]
    bg = background_color(grid)
    R, C = len(grid), len(grid[0])
    objs = seg_fn(grid)
    if not objs: return [row[:] for row in grid]
    result = [[bg] * C for _ in range(R)]

    if direction == "down":
        sorted_objs = sorted(objs, key=lambda o: -o.bbox()[2])
        occupied: set[tuple[int, int]] = set()
        for obj in sorted_objs:
            max_shift = R
            for r, c in obj.cells:
                for shift in range(1, R):
                    nr = r + shift
                    if nr >= R or (nr, c) in occupied:
                        max_shift = min(max_shift, shift - 1)
                        break
                else:
                    max_shift = min(max_shift, R - 1 - r)
            for r, c in obj.cells:
                new_r = r + max_shift
                occupied.add((new_r, c))
                result[new_r][c] = grid[r][c]

    elif direction == "up":
        sorted_objs = sorted(objs, key=lambda o: o.bbox()[0])
        occupied: set[tuple[int, int]] = set()
        for obj in sorted_objs:
            max_shift = R
            for r, c in obj.cells:
                for shift in range(1, R):
                    nr = r - shift
                    if nr < 0 or (nr, c) in occupied:
                        max_shift = min(max_shift, shift - 1)
                        break
                else:
                    max_shift = min(max_shift, r)
            for r, c in obj.cells:
                new_r = r - max_shift
                occupied.add((new_r, c))
                result[new_r][c] = grid[r][c]

    elif direction == "left":
        sorted_objs = sorted(objs, key=lambda o: o.bbox()[1])
        occupied: set[tuple[int, int]] = set()
        for obj in sorted_objs:
            max_shift = C
            for r, c in obj.cells:
                for shift in range(1, C):
                    nc = c - shift
                    if nc < 0 or (r, nc) in occupied:
                        max_shift = min(max_shift, shift - 1)
                        break
                else:
                    max_shift = min(max_shift, c)
            for r, c in obj.cells:
                new_c = c - max_shift
                occupied.add((r, new_c))
                result[r][new_c] = grid[r][c]

    elif direction == "right":
        sorted_objs = sorted(objs, key=lambda o: -o.bbox()[3])
        occupied: set[tuple[int, int]] = set()
        for obj in sorted_objs:
            max_shift = C
            for r, c in obj.cells:
                for shift in range(1, C):
                    nc = c + shift
                    if nc >= C or (r, nc) in occupied:
                        max_shift = min(max_shift, shift - 1)
                        break
                else:
                    max_shift = min(max_shift, C - 1 - c)
            for r, c in obj.cells:
                new_c = c + max_shift
                occupied.add((r, new_c))
                result[r][new_c] = grid[r][c]

    return result


def _count_objects_as_grid(grid: Grid, seg_fn: Callable[[Grid], list[Obj]]) -> Grid:
    objs = seg_fn(grid)
    n = len(objs)
    if n == 0:
        return [[0]]
    return [[1] * n]


# ---------------------------------------------------------------------------
# Op generation (~117 closures)
# ---------------------------------------------------------------------------

def generate_object_ops() -> dict[str, Callable[[Grid], Grid]]:
    """Generate all object-parameterized ops as Grid->Grid closures."""
    ops: dict[str, Callable[[Grid], Grid]] = {}

    for seg_name, seg_fn in SEGMENTATIONS.items():
        for sel_name, sel_fn in SELECTORS.items():

            def make_keep_only(sf=seg_fn, slct=sel_fn):
                def op(grid):
                    objs = sf(grid); obj = slct(objs)
                    if obj is None: return [row[:] for row in grid]
                    return _action_keep_only(grid, obj, sf)
                return op
            ops[f"obj_{seg_name}_{sel_name}_keep_only"] = make_keep_only()

            def make_remove(sf=seg_fn, slct=sel_fn):
                def op(grid):
                    objs = sf(grid); obj = slct(objs)
                    if obj is None: return [row[:] for row in grid]
                    return _action_remove(grid, obj, sf)
                return op
            ops[f"obj_{seg_name}_{sel_name}_remove"] = make_remove()

            def make_crop_to(sf=seg_fn, slct=sel_fn):
                def op(grid):
                    objs = sf(grid); obj = slct(objs)
                    if obj is None: return [row[:] for row in grid]
                    return _action_crop_to(grid, obj, sf)
                return op
            ops[f"obj_{seg_name}_{sel_name}_crop_to"] = make_crop_to()

            def make_recolor(sf=seg_fn, slct=sel_fn):
                def op(grid):
                    objs = sf(grid); obj = slct(objs)
                    if obj is None: return [row[:] for row in grid]
                    return _action_recolor_to_dominant(grid, obj, sf)
                return op
            ops[f"obj_{seg_name}_{sel_name}_recolor_to_dominant"] = make_recolor()

        def make_recolor_by_size(sf=seg_fn):
            def op(grid): return _recolor_by_size_rank(grid, sf)
            return op
        ops[f"obj_{seg_name}_recolor_by_size_rank"] = make_recolor_by_size()

        def make_gravity(sf=seg_fn, direction="down"):
            def op(grid): return _move_all_to_gravity(grid, sf, direction)
            return op
        ops[f"obj_{seg_name}_move_gravity_down"] = make_gravity(seg_fn, "down")
        ops[f"obj_{seg_name}_move_gravity_up"] = make_gravity(seg_fn, "up")
        ops[f"obj_{seg_name}_move_gravity_left"] = make_gravity(seg_fn, "left")
        ops[f"obj_{seg_name}_move_gravity_right"] = make_gravity(seg_fn, "right")

        def make_count(sf=seg_fn):
            def op(grid): return _count_objects_as_grid(grid, sf)
            return op
        ops[f"obj_{seg_name}_count_objects_as_grid"] = make_count()

    return ops


# ---------------------------------------------------------------------------
# Segmentation-consistency gate
# ---------------------------------------------------------------------------

def _seg_signature(grid: Grid, seg_fn: Callable[[Grid], list[Obj]]) -> tuple[int, frozenset[int]]:
    objs = seg_fn(grid)
    return (len(objs), frozenset(o.color for o in objs))


def filter_segmentations(demo_pairs: list[tuple[Grid, Grid]], threshold: float = 0.5) -> set[str]:
    """Return segmentation names to keep for this task."""
    if not demo_pairs:
        return set(SEGMENTATIONS.keys())
    kept: set[str] = set()
    for seg_name, seg_fn in SEGMENTATIONS.items():
        score = 0.0
        for inp, out in demo_pairs:
            in_count, in_colors = _seg_signature(inp, seg_fn)
            out_count, out_colors = _seg_signature(out, seg_fn)
            both_nontrivial = in_count > 0 and out_count > 0
            color_overlap = bool(in_colors & out_colors)
            same_count = (in_count == out_count) and in_count > 0
            close_count = both_nontrivial and abs(in_count - out_count) <= 2
            if both_nontrivial and (color_overlap or same_count or close_count):
                score += 1.0
        frac = score / len(demo_pairs)
        if frac >= threshold:
            kept.add(seg_name)
    if not kept:
        return set(SEGMENTATIONS.keys())
    return kept


def filter_object_ops(
    ops: dict[str, Callable[[Grid], Grid]],
    kept_segs: set[str],
) -> dict[str, Callable[[Grid], Grid]]:
    """Return only the ops belonging to kept segmentations."""
    filtered: dict[str, Callable[[Grid], Grid]] = {}
    for name, fn in ops.items():
        if not name.startswith("obj_"):
            filtered[name] = fn
            continue
        rest = name[4:]
        seg = rest.split("_")[0]
        if seg in kept_segs:
            filtered[name] = fn
    return filtered


print(f"objects OK — {len(generate_object_ops())} object ops generated")

In [ ]:
# ============================================================
# solvers — rung-1 trivial solver
# ============================================================

def _constant_demo_output(task: Task) -> Grid | None:
    outputs = [tuple(map(tuple, p.output)) for p in task.train]
    if len(set(outputs)) == 1:
        return [list(row) for row in outputs[0]]
    return None


def _majority_output_shape(task: Task) -> tuple[int, int]:
    shapes = Counter((len(p.output), len(p.output[0])) for p in task.train)
    return shapes.most_common(1)[0][0]


def solve_trivial(task: Task) -> list[tuple[Grid, Grid]]:
    """Attempt 1: identity. Attempt 2: constant demo output or zero-grid."""
    constant = _constant_demo_output(task)
    rows, cols = _majority_output_shape(task)
    fallback: Grid = [[0] * cols for _ in range(rows)]
    predictions: list[tuple[Grid, Grid]] = []
    for test_input in task.test_inputs:
        attempt_1 = [row[:] for row in test_input]
        attempt_2 = constant if constant is not None else fallback
        predictions.append((attempt_1, attempt_2))
    return predictions


print("solvers OK")

In [ ]:
# ============================================================
# search — BFS program search + solve_dsl_obj
# ============================================================

Program = tuple[str, ...]


def _grid_key(grid: Grid) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(row) for row in grid)


def _apply_seq(
    ops_seq: Program,
    grid: Grid,
    op_callables: dict[str, Callable[[Grid], Grid]],
) -> Grid:
    result = grid
    for name in ops_seq:
        result = op_callables[name](result)
    return result


def _predicted_shape_after_seq(ops_seq: Program, in_shape: tuple[int, int]) -> tuple[int, int] | None:
    r, c = in_shape
    for name in ops_seq:
        rule = SHAPE_RULES.get(name)
        if rule is None:
            return None
        result = rule(r, c)
        if result is None:
            return None
        r, c = result
    return (r, c)


def _shape_gate_ok(ops_seq: Program, demo_pairs: list[tuple[Grid, Grid]]) -> bool:
    if not demo_pairs:
        return True
    inp, out = demo_pairs[0]
    in_shape = (len(inp), len(inp[0]) if inp else 0)
    out_shape = (len(out), len(out[0]) if out else 0)
    pred = _predicted_shape_after_seq(ops_seq, in_shape)
    if pred is None:
        return True
    return pred == out_shape


def search_programs(task: Task, max_depth: int = 3) -> list[Program]:
    """Find all programs consistent with every demo pair (iterative-deepening BFS)."""
    demo_pairs: list[tuple[Grid, Grid]] = [
        (pair.input, pair.output) for pair in task.train
    ]
    if not demo_pairs:
        return []

    color_map_fn = build_color_map_op(demo_pairs)
    effective_ops: dict[str, Callable[[Grid], Grid]] = dict(OPS)
    effective_ops["infer_color_map"] = color_map_fn

    op_names = list(effective_ops.keys())
    first_inp, first_out = demo_pairs[0]
    first_out_key = _grid_key(first_out)
    consistent_programs: list[Program] = []

    for depth in range(1, max_depth + 1):
        current_layer: list[tuple[tuple[str, ...], Grid]] = [((), first_inp)]
        seen: list[set[tuple[tuple[int, ...], ...]]] = [set() for _ in range(depth + 1)]
        seen[0].add(_grid_key(first_inp))

        for step in range(depth):
            next_layer: list[tuple[tuple[str, ...], Grid]] = []
            next_seen = seen[step + 1]

            for prefix, inter_grid in current_layer:
                for op_name in op_names:
                    new_prefix = prefix + (op_name,)
                    if step == depth - 1:
                        if not _shape_gate_ok(new_prefix, demo_pairs):
                            continue
                    new_grid = effective_ops[op_name](inter_grid)
                    new_key = _grid_key(new_grid)
                    if new_key in next_seen:
                        continue
                    next_seen.add(new_key)
                    if step == depth - 1:
                        if new_key != first_out_key:
                            continue
                        all_match = True
                        for inp, out in demo_pairs[1:]:
                            result = _apply_seq(new_prefix, inp, effective_ops)
                            if _grid_key(result) != _grid_key(out):
                                all_match = False
                                break
                        if all_match:
                            consistent_programs.append(new_prefix)
                    else:
                        next_layer.append((new_prefix, new_grid))

            current_layer = next_layer

    return consistent_programs


def _is_object_op(name: str) -> bool:
    return name.startswith("obj_")


def _search_obj_programs(
    task: Task,
    effective_ops: dict[str, Callable[[Grid], Grid]],
    time_budget: float = 4.0,
) -> list[Program]:
    """Search with family-aware depth budget: depth 1-2 full, depth 3 restricted patterns."""
    import time as _time

    demo_pairs: list[tuple[Grid, Grid]] = [
        (pair.input, pair.output) for pair in task.train
    ]
    if not demo_pairs:
        return []

    op_names = list(effective_ops.keys())
    grid_op_names = [n for n in op_names if not _is_object_op(n)]
    obj_op_names = [n for n in op_names if _is_object_op(n)]

    first_inp, first_out = demo_pairs[0]
    first_out_key = _grid_key(first_out)
    consistent_programs: list[Program] = []
    t_start = _time.monotonic()

    def _timed_out() -> bool:
        return _time.monotonic() - t_start > time_budget

    def _verify_all(prefix: Program) -> bool:
        for inp, out in demo_pairs[1:]:
            result = _apply_seq(prefix, inp, effective_ops)
            if _grid_key(result) != _grid_key(out):
                return False
        return True

    for depth in range(1, 3):
        if _timed_out():
            return consistent_programs
        current_layer: list[tuple[tuple[str, ...], Grid]] = [((), first_inp)]
        seen: list[set[tuple[tuple[int, ...], ...]]] = [set() for _ in range(depth + 1)]
        seen[0].add(_grid_key(first_inp))
        for step in range(depth):
            if _timed_out():
                return consistent_programs
            next_layer: list[tuple[tuple[str, ...], Grid]] = []
            next_seen = seen[step + 1]
            eval_count = 0
            for prefix, inter_grid in current_layer:
                for op_name in op_names:
                    eval_count += 1
                    if eval_count % 500 == 0 and _timed_out():
                        return consistent_programs
                    new_prefix = prefix + (op_name,)
                    if step == depth - 1:
                        if not _shape_gate_ok(new_prefix, demo_pairs):
                            continue
                    new_grid = effective_ops[op_name](inter_grid)
                    new_key = _grid_key(new_grid)
                    if new_key in next_seen:
                        continue
                    next_seen.add(new_key)
                    if step == depth - 1:
                        if new_key != first_out_key:
                            continue
                        if _verify_all(new_prefix):
                            consistent_programs.append(new_prefix)
                    else:
                        next_layer.append((new_prefix, new_grid))
            current_layer = next_layer

    def _search_pattern(name_sets: list[list[str]]) -> bool:
        current: list[tuple[tuple[str, ...], Grid]] = [((), first_inp)]
        step_seen: list[set[tuple[tuple[int, ...], ...]]] = [set() for _ in range(4)]
        step_seen[0].add(_grid_key(first_inp))
        for step, allowed_names in enumerate(name_sets):
            if _timed_out():
                return False
            nxt: list[tuple[tuple[str, ...], Grid]] = []
            nxt_seen = step_seen[step + 1]
            is_last = (step == len(name_sets) - 1)
            eval_count = 0
            for prefix, inter_grid in current:
                for op_name in allowed_names:
                    eval_count += 1
                    if eval_count % 200 == 0 and _timed_out():
                        return False
                    new_prefix = prefix + (op_name,)
                    new_grid = effective_ops[op_name](inter_grid)
                    new_key = _grid_key(new_grid)
                    if new_key in nxt_seen:
                        continue
                    nxt_seen.add(new_key)
                    if is_last:
                        if new_key != first_out_key:
                            continue
                        if _verify_all(new_prefix):
                            consistent_programs.append(new_prefix)
                    else:
                        nxt.append((new_prefix, new_grid))
            current = nxt
        return True

    if not _timed_out():
        _search_pattern([grid_op_names, obj_op_names, grid_op_names])  # WOW
    if not _timed_out():
        _search_pattern([obj_op_names, grid_op_names, grid_op_names])  # OWW
    if not _timed_out():
        _search_pattern([grid_op_names, grid_op_names, obj_op_names])  # WWO

    return consistent_programs


def _make_pass2_predictions(
    task: Task,
    programs: list[Program],
    effective_ops: dict[str, Callable[[Grid], Grid]],
    fallback_preds: list[tuple[Grid, Grid]],
    solver_label: str,
) -> list[tuple[Grid, Grid]]:
    predictions: list[tuple[Grid, Grid]] = []
    for test_idx, test_input in enumerate(task.test_inputs):
        prog_predictions: list[tuple[Grid, Program]] = []
        for prog in programs:
            pred = _apply_seq(prog, test_input, effective_ops)
            prog_predictions.append((pred, prog))
        groups: dict[tuple[tuple[int, ...], ...], list[Program]] = {}
        group_grid: dict[tuple[tuple[int, ...], ...], Grid] = {}
        for pred_grid, prog in prog_predictions:
            key = _grid_key(pred_grid)
            if key not in groups:
                groups[key] = []
                group_grid[key] = pred_grid
            groups[key].append(prog)

        def group_sort_key(k):
            progs = groups[k]
            min_len = min(len(p) for p in progs)
            return (min_len, -len(progs))

        sorted_keys = sorted(groups.keys(), key=group_sort_key)
        attempt_1 = group_grid[sorted_keys[0]]
        attempt_2 = (
            group_grid[sorted_keys[1]]
            if len(sorted_keys) >= 2
            else fallback_preds[test_idx][1]
        )
        predictions.append((attempt_1, attempt_2))
        winning_prog = groups[sorted_keys[0]][0]
        log.info("task %s test[%d] [%s]: program=%s", task.task_id, test_idx, solver_label, " -> ".join(winning_prog))
    return predictions


def solve_dsl_obj(task: Task) -> list[tuple[Grid, Grid]]:
    """Rung 2B: object-aware solver with family-aware depth budget."""
    import time as _time

    demo_pairs_raw = [(pair.input, pair.output) for pair in task.train]
    color_map_fn = build_color_map_op(demo_pairs_raw)
    effective_ops: dict[str, Callable[[Grid], Grid]] = dict(OPS)
    effective_ops["infer_color_map"] = color_map_fn

    all_obj_ops = generate_object_ops()
    kept_segs = filter_segmentations(demo_pairs_raw)
    gated_obj_ops = filter_object_ops(all_obj_ops, kept_segs)

    combined_ops: dict[str, Callable[[Grid], Grid]] = dict(effective_ops)
    combined_ops.update(gated_obj_ops)

    t0_total = _time.monotonic()
    TOTAL_BUDGET = 4.5

    programs = _search_obj_programs(task, combined_ops, time_budget=2.0)
    trivial_preds = solve_trivial(task)

    if not programs:
        remaining = TOTAL_BUDGET - (_time.monotonic() - t0_total)
        if remaining > 0.3:
            dsl_max_depth = 3 if remaining > 1.5 else 2
            dsl_programs = search_programs(task, max_depth=dsl_max_depth)
            if dsl_programs:
                return _make_pass2_predictions(task, dsl_programs, effective_ops, trivial_preds, "dsl_fallback")
        return trivial_preds

    return _make_pass2_predictions(task, programs, combined_ops, trivial_preds, "dsl_obj")


print("search OK")

In [ ]:
# ============================================================
# Data discovery — locate test challenges JSON
# ============================================================
import glob

# Kaggle may mount data under /kaggle/input/ or /kaggle/input/competitions/
# Use glob to find the file regardless of nesting level.
# Fall back to local data/ directory for dry-run testing.

candidates = glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)
if not candidates:
    # Local fallback for dry-run
    candidates = glob.glob("data/arc-agi_test_challenges.json") + \
                 glob.glob("../data/arc-agi_test_challenges.json")

if not candidates:
    raise FileNotFoundError("Could not find arc-agi_test_challenges.json — check mount")

TEST_CHALLENGES_PATH = Path(candidates[0])
print(f"Found test challenges: {TEST_CHALLENGES_PATH}")

# Show all mounted paths under /kaggle/input for debugging
mounted = sorted(glob.glob("/kaggle/input/**", recursive=True))[:20]
if mounted:
    print("Mounted paths (first 20):")
    for p in mounted:
        print(" ", p)
else:
    print("(no /kaggle/input — running locally)")

In [ ]:
# ============================================================
# Driver — load tasks, run solver, write submission.json
# ============================================================
import time as _wall

SUBMISSION_PATH = Path("/kaggle/working/submission.json")
# When running locally, write next to this notebook
if not Path("/kaggle/working").exists():
    SUBMISSION_PATH = Path("submission.json")

TIME_BUDGET_SECONDS = 8 * 3600  # 8 hours hard wall (should never trigger)

# Load test tasks (no solutions exist for the hidden test set)
raw = json.loads(TEST_CHALLENGES_PATH.read_text())
tasks: list[Task] = []
for task_id, body in raw.items():
    tasks.append(Task(
        task_id=task_id,
        train=[Pair(p["input"], p["output"]) for p in body["train"]],
        test_inputs=[t["input"] for t in body["test"]],
        test_outputs=None,  # hidden
    ))

print(f"Loaded {len(tasks)} test tasks")

# Run solver
attempts: dict[str, list[tuple[Grid, Grid]]] = {}
t0 = _wall.monotonic()
budget_exceeded = False

for i, task in enumerate(tasks):
    elapsed = _wall.monotonic() - t0
    if elapsed > TIME_BUDGET_SECONDS:
        log.warning("Time budget exceeded at task %d/%d — falling back to trivial for remaining", i, len(tasks))
        budget_exceeded = True

    try:
        if budget_exceeded:
            preds = solve_trivial(task)
        else:
            preds = solve_dsl_obj(task)
    except Exception as exc:
        log.warning("Task %s raised %s — using trivial fallback", task.task_id, exc)
        preds = solve_trivial(task)

    attempts[task.task_id] = preds

    if (i + 1) % 20 == 0 or i == len(tasks) - 1:
        log.info("Progress: %d/%d tasks (%.1fs elapsed)", i + 1, len(tasks), _wall.monotonic() - t0)

total_elapsed = _wall.monotonic() - t0
print(f"\nSolver done — {len(attempts)} tasks in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")

# Write submission
build_submission(attempts, SUBMISSION_PATH)
print(f"Submission written to: {SUBMISSION_PATH}")

In [ ]:
# ============================================================
# Validate submission and print summary
# ============================================================
submission = json.loads(SUBMISSION_PATH.read_text())

errors = []
for task_id, entries in submission.items():
    if not isinstance(entries, list):
        errors.append(f"{task_id}: entries is not a list")
        continue
    for idx, entry in enumerate(entries):
        for key in ("attempt_1", "attempt_2"):
            if key not in entry:
                errors.append(f"{task_id}[{idx}]: missing {key}")
                continue
            grid = entry[key]
            if not isinstance(grid, list) or not all(isinstance(row, list) for row in grid):
                errors.append(f"{task_id}[{idx}].{key}: not a list-of-lists")
                continue
            for r, row in enumerate(grid):
                for c, val in enumerate(row):
                    if not isinstance(val, int):
                        errors.append(f"{task_id}[{idx}].{key}[{r}][{c}]: not an int ({type(val)})")

# Check coverage
all_task_ids = set(raw.keys())
submitted_ids = set(submission.keys())
missing = all_task_ids - submitted_ids
if missing:
    errors.append(f"Missing {len(missing)} tasks from submission: {list(missing)[:5]}")

print(f"Tasks in submission:  {len(submission)}")
print(f"Tasks in test file:   {len(all_task_ids)}")
print(f"Coverage:             {'OK' if not missing else 'INCOMPLETE'}")
print(f"Validation errors:    {len(errors)}")
if errors:
    for e in errors[:10]:
        print(" ERROR:", e)
else:
    print("All validation checks passed.")

# Print a sample entry
sample_id = list(submission.keys())[0]
sample_entry = submission[sample_id]
print(f"\nSample entry ({sample_id}):")
print(f"  test inputs: {len(sample_entry)}")
for i, e in enumerate(sample_entry):
    a1 = e['attempt_1']
    a2 = e['attempt_2']
    print(f"  [{i}] attempt_1 shape={len(a1)}x{len(a1[0]) if a1 else 0}, attempt_2 shape={len(a2)}x{len(a2[0]) if a2 else 0}")

print(f"\nTotal elapsed: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")